# Présentation du notebook

Ce notebook est dédié à l'exploration du dataset Yelp utilisé dans le projet. Il permet de comprendre la structure des données et les relations entre les différentes entités.

# Objectif

Analyser les données avant leur utilisation dans la base multi-modèles ArangoDB.

In [1]:
from arango import ArangoClient
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
ARANGO_URL = "http://localhost:8529"
ARANGO_DB = "YelpDB"
ARANGO_USER = "root"
ARANGO_PASSWORD = "root"

In [3]:
client = ArangoClient(hosts=ARANGO_URL)

db = client.db(
    ARANGO_DB,
    username=ARANGO_USER,
    password=ARANGO_PASSWORD
)

print(f"Connexion à la base '{ARANGO_DB}' réussie ")

Connexion à la base 'YelpDB' réussie 


In [4]:
collections = [
    collection
    for collection in db.collections()
    if not collection["name"].startswith("_")
]

print(f"Nombre de collections : {len(collections)}")

Nombre de collections : 14


In [5]:
collections = [
    c for c in db.collections()
    if not c["name"].startswith("_")
]

collection_info = []

for c in collections:
    name = c["name"]
    type_value = str(c["type"]).lower()

    if type_value == "edge":
        collection_type = "EDGE"
    elif type_value == "document":
        collection_type = "DOCUMENT"
    else:
        collection_type = f"UNKNOWN ({type_value})"

    collection_info.append({
        "Collection": name,
        "Type": collection_type,
        "Nombre": db.collection(name).count()
    })

df_collections = pd.DataFrame(collection_info)

df_collections

,Collection,Type,Nombre
0,Reviews,DOCUMENT,1275792
1,Users,DOCUMENT,686110
2,WritesReview,EDGE,1274916
3,BusinessCategory,EDGE,511382
4,Businesses,DOCUMENT,85899
5,BusinessCheckin,EDGE,854658
6,Checkins,DOCUMENT,854686
7,Neighborhoods,DOCUMENT,72112
8,WritesTip,EDGE,647182
9,TipsBusiness,EDGE,647764


In [6]:
document_collections = [
    c["name"]
    for c in collections
    if str(c["type"]).lower() == "document"
]

edge_collections = [
    c["name"]
    for c in collections
    if str(c["type"]).lower() == "edge"
]

print("Collections DOCUMENT :")
for name in document_collections:
    print(" -", name)

print("\nCollections EDGE :")
for name in edge_collections:
    print(" -", name)

Collections DOCUMENT :
 - Reviews
 - Users
 - Businesses
 - Checkins
 - Neighborhoods
 - Categories
 - Tips

Collections EDGE :
 - WritesReview
 - BusinessCategory
 - BusinessCheckin
 - WritesTip
 - TipsBusiness
 - ReviewsBusiness
 - BusinessNeighborhood


In [7]:
def execute_aql(query, bind_vars=None):
    """
    Exécute une requête AQL et retourne les résultats sous forme de liste.
    """
    
    cursor = db.aql.execute(
        query,
        bind_vars=bind_vars or {}
    )

    return list(cursor)

In [8]:
for collection_name in document_collections:

    print("=" * 70)
    print(f"Collection : {collection_name}")
    print("=" * 70)

    query = f"""
    FOR doc IN `{collection_name}`
        LIMIT 3
        RETURN doc
    """

    documents = execute_aql(query)

    display(pd.DataFrame(documents))

Collection : Reviews


,_key,_id,_rev,rid,business_id,user_id,rating,text,year,month
0,2464791,Reviews/2464791,_mE4ngQa---,20000,vIXUw1cLG84-m8LzOg3f-w,2Wni-wBq5rcJcbV2DFvyDg,2,Oy. Why do I do this to myself.\n\nThe servic...,2011,March
1,2464792,Reviews/2464792,_mE4ngQa--_,20001,vIXUw1cLG84-m8LzOg3f-w,ts7EG6Zv2zdMDg29nyqGfA,2,Shame on me for not consulting Yelp prior to g...,2011,July
2,2464793,Reviews/2464793,_mE4ngQa--A,20002,vIXUw1cLG84-m8LzOg3f-w,V1eKHG6AtJyru_HvwPanEg,1,I don't know who of my group picked this place...,2012,January


Collection : Users


,_key,_id,_rev,uid,user_id,name
0,3740770,Users/3740770,_mE4smPO---,20000,Ib5OiwzQ0D2sgkSLBQg3wQ,Rick
1,3740771,Users/3740771,_mE4smPO--_,20001,ylprSS079fpVZiwwNHdQLQ,Lilly
2,3740772,Users/3740772,_mE4smPO--A,20002,ZsmIoG8nDqIPQhCcviiP3g,Taylor


Collection : Businesses


...

,_key,_id,_rev,id,business_id,neighborhood_name
0,2392509,Neighborhoods/2392509,_mE4ktri---,20000,nJwzIiboAN8TFrIJUzakuQ,Pointe-Claire
1,2392510,Neighborhoods/2392510,_mE4ktri--_,20001,49Zmv7wIt853wMIKzAH-og,Southwest
2,2392511,Neighborhoods/2392511,_mE4ktri--A,20002,vkTZxQ69vF7FEROtBedfXg,Southeast


Collection : Categories


,_key,_id,_rev,id,business_id,category_name
0,1880981,Categories/1880981,_mE4iV5S---,20000,kmiYqdQYJ5JmQQHgFdHCug,Venues & Event Spaces
1,1880982,Categories/1880982,_mE4iV5S--_,20001,kmiYqdQYJ5JmQQHgFdHCug,Hotels
2,1880983,Categories/1880983,_mE4iV5S--A,20002,wiI1M3c-R46pRhGYO1FHWQ,Pet Services


Collection : Tips


,_key,_id,_rev,tip_id,business_id,text,user_id,likes,year,month
0,378325,Tips/378325,_mE4edNG---,20000,rHtkiJVIT-AFexNcEQ1vYQ,Brandi is awesome!,nzJbI_NNZEq5YqcQbyz9Eg,0,2011,December
1,378326,Tips/378326,_mE4edNG--_,20001,rHtkiJVIT-AFexNcEQ1vYQ,Chop chop,3dno9Y7mtvi2tq6IfMpDeA,0,2012,January
2,378327,Tips/378327,_mE4edNG--A,20002,rHtkiJVIT-AFexNcEQ1vYQ,Got a awesome perm from Brandi last summer and...,qyaQ9RxJu9k_hJQkvsC5-w,0,2012,February


In [9]:
for edge_name in edge_collections:

    print("=" * 70)
    print(f"EDGE : {edge_name}")
    print("=" * 70)

    query = f"""
    FOR e IN `{edge_name}`
        LIMIT 5
        RETURN e
    """

    edges = execute_aql(query)

    display(pd.DataFrame(edges))

EDGE : WritesReview


,_key,_id,_from,_to,_rev
0,c04671f7e280c72780ae0fd4ff27e4ff43292289,WritesReview/c04671f7e280c72780ae0fd4ff27e4ff4...,Users/3740771,Reviews/2621525,_mFCOs8u---
1,5ef74d2b812139aca393c787fa1c25f825212789,WritesReview/5ef74d2b812139aca393c787fa1c25f82...,Users/3740772,Reviews/2952616,_mFCOs8u--_
2,49598ece1aa379e94bfbe4fcb5131b98f208bb2e,WritesReview/49598ece1aa379e94bfbe4fcb5131b98f...,Users/3740772,Reviews/2957717,_mFCOs8u--A
3,c23cc9968293b79712afb4783c9000a6b1f4591a,WritesReview/c23cc9968293b79712afb4783c9000a6b...,Users/3740775,Reviews/2876290,_mFCOs8u--B
4,b30f4330a7a2040b1e2fb6dd02db49676c8f9bb0,WritesReview/b30f4330a7a2040b1e2fb6dd02db49676...,Users/3740776,Reviews/2601997,_mFCOs8u--C


EDGE : BusinessCategory


,_key,_id,_from,_to,_rev
0,e0a01426944544a59dc19db1fa1af9c9f0fe2815,BusinessCategory/e0a01426944544a59dc19db1fa1af...,Businesses/279088,Categories/1880981,_mFC137i---
1,a16ca5f3be76f9fcf50315c236d0dd91c6210220,BusinessCategory/a16ca5f3be76f9fcf50315c236d0d...,Businesses/279088,Categories/1880982,_mFC137m---
2,d7d86d74ab3d1c468f92b2d454c09d42ce9f1401,BusinessCategory/d7d86d74ab3d1c468f92b2d454c09...,Businesses/279089,Categories/1880983,_mFC137m--_
3,0f38dd3e999f24c188bf5d43dae45db3067e9669,BusinessCategory/0f38dd3e999f24c188bf5d43dae45...,Businesses/279089,Categories/1880984,_mFC137m--A
4,28127e1967c906bee87039b6ec8ac7c783fec3ee,BusinessCategory/28127e1967c906bee87039b6ec8ac...,Businesses/279089,Categories/1880985,_mFC137m--B


EDGE : BusinessCheckin


...

,_key,_id,_from,_to,_rev
0,cdc7687c0a9a470093c1751679e563b02b04e4c0,TipsBusiness/cdc7687c0a9a470093c1751679e563b02...,Tips/378325,Businesses/275881,_mFC09kK---
1,ef47dd21d1fd977734c42003c817c7a4c2a74703,TipsBusiness/ef47dd21d1fd977734c42003c817c7a4c...,Tips/378326,Businesses/275881,_mFC09kK--_
2,7266f339cd6c0656b384ecbc678b1685a1dd26df,TipsBusiness/7266f339cd6c0656b384ecbc678b1685a...,Tips/378327,Businesses/275881,_mFC09kK--A
3,8884e5d76d6732fd7831604bd5c2aca9a5a9c36e,TipsBusiness/8884e5d76d6732fd7831604bd5c2aca9a...,Tips/378328,Businesses/275881,_mFC09kK--B
4,44e456bc3b7fdb68e06bcbbb2d7edc9ef2a3eafb,TipsBusiness/44e456bc3b7fdb68e06bcbbb2d7edc9ef...,Tips/378329,Businesses/275881,_mFC09kK--C


EDGE : ReviewsBusiness


,_key,_id,_from,_to,_rev
0,8b870dcc4bc69422a9df846e78c07e74a8075c82,ReviewsBusiness/8b870dcc4bc69422a9df846e78c07e...,Reviews/2464791,Businesses/273309,_mFCyMhS---
1,06046098922ec370298ef6a53533d249047ce7ea,ReviewsBusiness/06046098922ec370298ef6a53533d2...,Reviews/2464792,Businesses/273309,_mFCyMhS--_
2,bdfd5127e0435b93c97611ec4a8ea8b52b5fc470,ReviewsBusiness/bdfd5127e0435b93c97611ec4a8ea8...,Reviews/2464793,Businesses/273309,_mFCyMhS--A
3,930ef5e632b6e5a145a9858f628b93fb4b1a5e5f,ReviewsBusiness/930ef5e632b6e5a145a9858f628b93...,Reviews/2464794,Businesses/273309,_mFCyMhS--B
4,95d6678a8d3dbb79253f2e9e0fb53fe212cdbfba,ReviewsBusiness/95d6678a8d3dbb79253f2e9e0fb53f...,Reviews/2464795,Businesses/273309,_mFCyMhS--C


EDGE : BusinessNeighborhood


,_key,_id,_from,_to,_rev
0,69cd349ff3c6caa726d406221972bfa83474e786,BusinessNeighborhood/69cd349ff3c6caa726d406221...,Businesses/235280,Neighborhoods/2392509,_mFC38-u---
1,fc84abfad2c0b89663b6e65ff60f28a02303a499,BusinessNeighborhood/fc84abfad2c0b89663b6e65ff...,Businesses/235281,Neighborhoods/2392510,_mFC38-u--_
2,72e5217708613dca31c4bee2e14982892d553eca,BusinessNeighborhood/72e5217708613dca31c4bee2e...,Businesses/235283,Neighborhoods/2392511,_mFC38-u--A
3,59dfbbbac32ef1a74d212bdf3fce0b93c71c3a25,BusinessNeighborhood/59dfbbbac32ef1a74d212bdf3...,Businesses/235284,Neighborhoods/2392512,_mFC38-u--B
4,52da5beb617066f08b6a2d9048cd5b81a97bcc6c,BusinessNeighborhood/52da5beb617066f08b6a2d904...,Businesses/235286,Neighborhoods/2392513,_mFC38-y---


## Conclusion

L’exploration du dataset Yelp a permis de construire une base ArangoDB multi-modèle composée de **sept collections documentaires** : `Businesses`, `Users`, `Reviews`, `Tips`, `Categories`, `Checkins` et `Neighborhoods`, ainsi que de **sept collections d’arêtes** : `WritesReview`, `ReviewsBusiness`, `WritesTip`, `TipsBusiness`, `BusinessCategory`, `BusinessCheckin` et `BusinessNeighborhood`.

Ces collections et leurs relations, regroupées dans le graphe `YelpGraph`, permettent de réaliser des requêtes documentaires et des traversées de graphe en AQL. Cette structure constitue la base utilisée pour la construction des requêtes **Gold AQL** et le développement du système **Text-to-AQL**.